# Open Door Legal — Client Feedback Analysis

Analysis of 324 client feedback responses covering:
- **Net Promoter Score & quantitative ratings**
- **Breakdowns** by case owner and degree of resolution
- **Trends over time**
- **Topic modeling** — what clients say ODL does well vs. what could improve (multilingual, auto-translated)

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn nltk openpyxl deep-translator -q
import nltk
nltk.download('stopwords', quiet=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from deep_translator import GoogleTranslator

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

STOP_WORDS = set(stopwords.words('english'))
ODL_STOPS  = {'open', 'door', 'legal', 'odl', 'help', 'helped',
              'would', 'could', 'also', 'one', 'really', 'think',
              'feel', 'know', 'get', 'got', 'make', 'made', 'like',
              'need', 'needed', 'nothing', 'everything', 'satisfied'}
ALL_STOPS  = STOP_WORDS | ODL_STOPS

## 1. Load Data

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/report_feedback_filtered.xlsx')

df = df.rename(columns={
    'Client Feedback: Created Date':           'date',
    'Case: Client Name':                       'client_name',
    'Case: Case Number':                       'case_number',
    'Case: Subject':                           'case_subject',
    'Case: Degree of Resolution':              'resolution',
    'Case: Case Owner':                        'case_owner',
    'Net Promoter Score':                      'nps',
    'How much of a positive diff has ODL had': 'positive_diff',
    'What does Open Door Legal do well':       'do_well',
    'What could Open Door Legal do better':    'do_better',
    'How well has ODL met your needs':         'needs_met',
    'Informed About Case':                     'informed',
    'Did we give choices':                     'gave_choices',
    'Barriers to services':                    'barriers',
    'Anything we could not have helped with?': 'beyond_scope',
})

df['date'] = pd.to_datetime(df['date'], errors='coerce')

print(f'Loaded {len(df):,} responses')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
df.head(3)

## 2. Quantitative Overview

In [ ]:
# NPS categories
def nps_category(score):
    if score >= 9:  return 'Promoter'
    if score >= 7:  return 'Passive'
    return 'Detractor'

df['nps_category'] = df['nps'].apply(nps_category)

nps_score = (
    (df['nps_category'] == 'Promoter').mean() -
    (df['nps_category'] == 'Detractor').mean()
) * 100

print(f'Net Promoter Score: {nps_score:.1f}')
print(df['nps_category'].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# NPS score distribution
sns.histplot(df['nps'], bins=10, discrete=True, ax=axes[0],
             color='steelblue', edgecolor='white')
axes[0].set_title('Net Promoter Score Distribution')
axes[0].set_xlabel('Score (0–10)')
axes[0].set_ylabel('Number of clients')
for x, color in [(range(0,7),'tomato'), (range(7,9),'gold'), (range(9,11),'mediumseagreen')]:
    for xi in x:
        axes[0].axvspan(xi-0.5, xi+0.5, alpha=0.08, color=color)

# NPS category breakdown
cat_counts = df['nps_category'].value_counts()
colors = {'Promoter':'mediumseagreen','Passive':'gold','Detractor':'tomato'}
bars = axes[1].bar(cat_counts.index, cat_counts.values,
                   color=[colors[c] for c in cat_counts.index])
axes[1].bar_label(bars, fmt='%d', padding=3)
axes[1].set_title(f'NPS Categories  (NPS = {nps_score:.0f})')
axes[1].set_ylabel('Count')

plt.suptitle('Net Promoter Score', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_nps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Positive difference
pos_diff = df['positive_diff'].value_counts()
order = ['Extreme','High','Moderate','Low','None']
order = [o for o in order if o in pos_diff.index]
pd_colors = {'Extreme':'#2ecc71','High':'#82e0aa','Moderate':'#f4d03f',
             'Low':'#e59866','None':'#e74c3c'}
axes[0].bar(order, [pos_diff.get(o, 0) for o in order],
            color=[pd_colors.get(o,'steelblue') for o in order])
axes[0].set_title('How much of a positive difference\nhas ODL had?')
axes[0].set_ylabel('Count')
for i, o in enumerate(order):
    v = pos_diff.get(o, 0)
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=9)

# Degree of resolution
res = df['resolution'].value_counts().dropna()
res_colors = {'Positive':'mediumseagreen','Neutral':'steelblue','Negative':'tomato'}
axes[1].bar(res.index, res.values,
            color=[res_colors.get(r,'grey') for r in res.index])
axes[1].set_title('Degree of Resolution')
axes[1].set_ylabel('Count')
for i, (idx, v) in enumerate(res.items()):
    axes[1].text(i, v + 1, str(v), ha='center', fontsize=9)

plt.suptitle('Client Satisfaction Ratings', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_ratings.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Yes/No questions
questions = {
    'Informed about case': df['informed'],
    'Gave choices':        df['gave_choices'],
    'Barriers to service': df['barriers'],
    'Beyond scope':        df['beyond_scope'],
}

fig, axes = plt.subplots(1, len(questions), figsize=(14, 4))
for ax, (label, col) in zip(axes, questions.items()):
    counts = col.value_counts()
    colors = ['mediumseagreen' if i == 'Yes' else 'tomato' if i == 'No' else 'steelblue'
              for i in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors)
    ax.bar_label(bars, fmt='%d', padding=2)
    total = counts.sum()
    pct = (counts / total * 100).round(1)
    ax.set_title(f'{label}\n({pct.get("Yes",0):.0f}% Yes)', fontsize=10)
    ax.set_ylabel('Count' if ax == axes[0] else '')

plt.suptitle('Yes / No Questions', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_yesno.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# NPS by case owner (owners with 5+ responses)
owner_counts = df['case_owner'].value_counts()
active_owners = owner_counts[owner_counts >= 5].index

owner_nps = (
    df[df['case_owner'].isin(active_owners)]
    .groupby('case_owner')['nps']
    .agg(['mean','count'])
    .round(2)
    .sort_values('mean', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(owner_nps.index, owner_nps['mean'], color='steelblue')
ax.axvline(df['nps'].mean(), color='tomato', linestyle='--', label=f'Overall mean ({df["nps"].mean():.1f})')
for i, (idx, row) in enumerate(owner_nps.iterrows()):
    ax.text(row['mean'] + 0.05, i, f"{row['mean']:.1f} (n={int(row['count'])})", va='center', fontsize=8)
ax.set_title('Average NPS by Case Owner (≥5 responses)')
ax.set_xlabel('Mean NPS')
ax.set_xlim(0, 11)
ax.legend()
plt.tight_layout()
plt.savefig('fig_nps_by_owner.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Translate Responses to English

In [ ]:
def translate(text) -> str:
    if pd.isna(text) or str(text).strip() in ('', 'nan', 'N/A', 'n/a', 'No', 'None'):
        return ''
    try:
        return GoogleTranslator(source='auto', target='en').translate(str(text))
    except Exception:
        return str(text)

print('Translating "do well" responses...')
df['do_well_en'] = df['do_well'].apply(translate)
print('Translating "do better" responses...')
df['do_better_en'] = df['do_better'].apply(translate)
print('Done.')

In [ ]:
def clean_text(text) -> str:
    if pd.isna(text) or str(text).strip() == '':
        return ''
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [t for t in text.split() if t not in ALL_STOPS and len(t) > 2]
    return ' '.join(tokens)

df['do_well_clean']   = df['do_well_en'].apply(clean_text)
df['do_better_clean'] = df['do_better_en'].apply(clean_text)

dw = df[df['do_well_clean'].str.len() > 0].copy()
db = df[df['do_better_clean'].str.len() > 0].copy()

print(f'"Do well" responses with text:   {len(dw)}')
print(f'"Do better" responses with text: {len(db)}')

# Spot-check a few translations
df[['do_well','do_well_en']].dropna().sample(3)

## 5. Topic Modeling — What ODL Does Well

In [ ]:
N_TOPICS_WELL = 5
N_TOP_WORDS   = 8

count_vec_dw = CountVectorizer(max_features=3000, min_df=2, ngram_range=(1,2))
count_mat_dw = count_vec_dw.fit_transform(dw['do_well_clean'])
vocab_dw     = count_vec_dw.get_feature_names_out()

lda_dw = LatentDirichletAllocation(
    n_components=N_TOPICS_WELL, random_state=42,
    learning_method='batch', max_iter=30
)
lda_dw.fit(count_mat_dw)

doc_topics_dw = lda_dw.transform(count_mat_dw)
dw['topic']        = doc_topics_dw.argmax(axis=1)
dw['topic_weight'] = doc_topics_dw.max(axis=1)

def top_words_table(model, vocab, n=8):
    rows = []
    for i, comp in enumerate(model.components_):
        words = [vocab[j] for j in comp.argsort()[:-n-1:-1]]
        rows.append({'Topic': f'Topic {i}', 'Top Words': ', '.join(words)})
    return pd.DataFrame(rows)

print('Topics — What ODL does well:')
display(top_words_table(lda_dw, vocab_dw, N_TOP_WORDS))

In [ ]:
# ── Label topics after reviewing the top words above ──────────────────────
# Edit these to match what you see in the topic output
TOPIC_LABELS_WELL = {
    0: 'Topic 0',
    1: 'Topic 1',
    2: 'Topic 2',
    3: 'Topic 3',
    4: 'Topic 4',
}
dw['topic_label'] = dw['topic'].map(TOPIC_LABELS_WELL)

In [ ]:
topic_counts_dw = dw['topic_label'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(topic_counts_dw.index, topic_counts_dw.values, color='mediumseagreen')
ax.bar_label(bars, fmt='%d', padding=3)
ax.set_title('What ODL Does Well — Topic Prevalence')
ax.set_xlabel('Number of responses')
plt.tight_layout()
plt.savefig('fig_topics_well.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Topic Modeling — What Could Be Better

In [ ]:
N_TOPICS_BETTER = 5

count_vec_db = CountVectorizer(max_features=3000, min_df=2, ngram_range=(1,2))
count_mat_db = count_vec_db.fit_transform(db['do_better_clean'])
vocab_db     = count_vec_db.get_feature_names_out()

lda_db = LatentDirichletAllocation(
    n_components=N_TOPICS_BETTER, random_state=42,
    learning_method='batch', max_iter=30
)
lda_db.fit(count_mat_db)

doc_topics_db = lda_db.transform(count_mat_db)
db['topic']        = doc_topics_db.argmax(axis=1)
db['topic_weight'] = doc_topics_db.max(axis=1)

print('Topics — What could be better:')
display(top_words_table(lda_db, vocab_db, N_TOP_WORDS))

In [ ]:
# ── Label topics after reviewing the top words above ──────────────────────
TOPIC_LABELS_BETTER = {
    0: 'Topic 0',
    1: 'Topic 1',
    2: 'Topic 2',
    3: 'Topic 3',
    4: 'Topic 4',
}
db['topic_label'] = db['topic'].map(TOPIC_LABELS_BETTER)

In [ ]:
topic_counts_db = db['topic_label'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(topic_counts_db.index, topic_counts_db.values, color='steelblue')
ax.bar_label(bars, fmt='%d', padding=3)
ax.set_title('What Could Be Better — Topic Prevalence')
ax.set_xlabel('Number of responses')
plt.tight_layout()
plt.savefig('fig_topics_better.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Topic Prevalence — Side by Side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

topic_counts_dw = dw['topic_label'].value_counts().sort_values(ascending=True)
topic_counts_db = db['topic_label'].value_counts().sort_values(ascending=True)

bars1 = axes[0].barh(topic_counts_dw.index, topic_counts_dw.values, color='mediumseagreen')
axes[0].bar_label(bars1, fmt='%d', padding=3)
axes[0].set_title('What ODL Does Well')
axes[0].set_xlabel('Number of responses')

bars2 = axes[1].barh(topic_counts_db.index, topic_counts_db.values, color='steelblue')
axes[1].bar_label(bars2, fmt='%d', padding=3)
axes[1].set_title('What Could Be Better')
axes[1].set_xlabel('Number of responses')

plt.suptitle('Topic Prevalence', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_topics_combined.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Trends Over Time

In [ ]:
monthly = (
    df.set_index('date')
    .resample('M')
    .agg(
        nps_mean=('nps', 'mean'),
        response_count=('nps', 'count')
    )
    .dropna()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(monthly.index, monthly['nps_mean'], marker='o', color='steelblue')
axes[0].axhline(monthly['nps_mean'].mean(), color='grey', linestyle='--', linewidth=0.8,
                label=f'Overall mean ({monthly["nps_mean"].mean():.1f})')
axes[0].set_ylabel('Mean NPS')
axes[0].set_title('Monthly NPS Trend')
axes[0].set_ylim(0, 10)
axes[0].legend()

axes[1].bar(monthly.index, monthly['response_count'], color='steelblue',
            width=20, alpha=0.7)
axes[1].set_ylabel('Number of responses')
axes[1].set_title('Monthly Response Volume')

plt.tight_layout()
plt.savefig('fig_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Export Results & Push to GitHub

In [ ]:
export_cols = [
    'date', 'client_name', 'case_number', 'case_owner', 'resolution',
    'nps', 'nps_category', 'positive_diff',
    'do_well', 'do_well_en',
    'do_better', 'do_better_en',
    'informed', 'gave_choices', 'barriers', 'beyond_scope'
]
export_cols = [c for c in export_cols if c in df.columns]
df[export_cols].to_csv('odl_feedback_results.csv', index=False)
print('Saved → odl_feedback_results.csv')

top_words_table(lda_dw, vocab_dw, N_TOP_WORDS).to_csv('topics_do_well.csv', index=False)
top_words_table(lda_db, vocab_db, N_TOP_WORDS).to_csv('topics_do_better.csv', index=False)
print('Saved → topics_do_well.csv')
print('Saved → topics_do_better.csv')

In [ ]:
!pip install PyGithub -q

from github import Github
from google.colab import userdata
from pathlib import Path

# ── Config ─────────────────────────────────────────────────────────────────
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set this in Colab's key icon sidebar
REPO_NAME    = 'RyAlah/ODL_Analysis'
BRANCH       = 'claude/sentiment-analysis-notebook-cn8ymo'
OUTPUT_DIR   = 'outputs'
# ───────────────────────────────────────────────────────────────────────────

files_to_push = [
    'odl_feedback_results.csv',
    'topics_do_well.csv',
    'topics_do_better.csv',
    'fig_nps.png',
    'fig_ratings.png',
    'fig_yesno.png',
    'fig_nps_by_owner.png',
    'fig_topics_well.png',
    'fig_topics_better.png',
    'fig_topics_combined.png',
    'fig_trends.png',
]

g    = Github(GITHUB_TOKEN)
repo = g.get_repo(REPO_NAME)

for fname in files_to_push:
    p = Path(fname)
    if not p.exists():
        print(f"Skipped (not found) → {fname}")
        continue
    content = p.read_bytes()
    path    = f"{OUTPUT_DIR}/{fname}"
    try:
        existing = repo.get_contents(path, ref=BRANCH)
        repo.update_file(path, f"Update {fname}", content, existing.sha, branch=BRANCH)
        print(f"Updated → {path}")
    except Exception:
        repo.create_file(path, f"Add {fname}", content, branch=BRANCH)
        print(f"Created → {path}")